In [1]:
import pandas as pd

In [2]:
survey_data = "Recommersion_10 febbraio 2025_16.17.csv"

In [3]:
df = pd.read_csv(survey_data).drop([0,1])
df

,StartDate,EndDate,Status,IPAddress,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,RecipientLastName,...,Custom_Cosine3_1,Custom_Cosine3_2,Custom_Cosine3_3,Custom_Cosine3_4,Custom_Cosine3_5,Custom_Cosine3_6,Diff_cust_cos3_1,Diff_cust_cos3_2,Recommersion_utility,Description_utility
2,2025-02-10 15:46:05,2025-02-10 15:51:04,1,NaN,100,298,1,2025-02-10 15:51:04,R_8JwQm622E3Mx8Np,NaN,...,4,2,3,3,3,3,"0,139","0,188",4,Miao
3,2025-02-10 15:52:52,2025-02-10 15:56:07,1,NaN,100,194,1,2025-02-10 15:56:07,R_2qV9tpNKae0Buux,NaN,...,3,2,3,3,2,2,"0,279","0,296",4,Miao


Metrics:
- Mean Average Precision
- Mean Reciprocal Rank
- Normalized Discounted Cumulative Gain

Metrics for these specifics:
- Audeering Euclidean
- Audeering Cosine
- Custom Euclidean
- Custom Cosine


In [ ]:
import numpy as np
import math

emotion_number = 2
cut = 5

def calculate_map(model, spotify=False):
    def compute_ap(user):
        if spotify:
            return float(df[f"{model}3_1"].iloc[user]) / cut
        else:
            return np.mean([float(df[f"{model}{i+1}_1"].iloc[user]) / cut for i in range(emotion_number)])

    map_values = map(compute_ap, range(len(df)))
    return np.mean(list(map_values))

def calculate_ndcg(model, spotify=False):
    def calculate_dcg(index, value):
        return (2**int(value) - 1) / math.log2(index + 2)

    def compute_ndcg_emotion(user):
        def compute_dcg_for_emotion(e):
            dcg_values = [df[f"{model}{e+1}_{i+2}"].iloc[user] for i in range(cut)]
            idcg_values = sorted(dcg_values, reverse=True)
            dcg = sum(map(calculate_dcg, range(len(dcg_values)), dcg_values))
            idcg = sum(map(calculate_dcg, range(len(idcg_values)), idcg_values))
            return dcg / idcg if idcg > 0 else 0

        if spotify:
            dcg_values = [df[f"{model}3_{i+2}"].iloc[user] for i in range(cut)]
            idcg_values = sorted(dcg_values, reverse=True)
            dcg = sum(map(calculate_dcg, range(len(dcg_values)), dcg_values))
            idcg = sum(map(calculate_dcg, range(len(idcg_values)), idcg_values))
            return dcg / idcg if idcg > 0 else 0
        else:
            return np.mean([compute_dcg_for_emotion(e) for e in range(emotion_number)])

    ndcg_values = map(compute_ndcg_emotion, range(len(df)))
    return np.mean(list(ndcg_values))

def calculate_mrr(model, spotify=False):
    relevance = [4, 5]

    def compute_mrr_for_ranking(ranking):
        for i, v in enumerate(ranking):
            if int(v) in relevance:
                return 1 / (i + 1)
        return 0

    def compute_mrr_emotion(user):
        if spotify:
            ranking = [df[f"{model}3_{i+2}"].iloc[user] for i in range(cut)]
            return compute_mrr_for_ranking(ranking)
        else:
            rankings = [df[f"{model}{e+1}_{i+2}"].iloc[user] for e in range(emotion_number) for i in range(cut)]
            return np.mean([compute_mrr_for_ranking(rankings[i:i+cut]) for i in range(0, len(rankings), cut)])

    mrr_values = map(compute_mrr_emotion, range(len(df)))
    return np.mean(list(mrr_values))


In [7]:
print("MAP Audeering_Euclidean =", calculate_map("Audeering_Euclidean"))
print("MAP Audeering_Cosine =", calculate_map("Audeering_Cosine"))
print("MAP Custom_Euclidean =", calculate_map("Custom_Euclidean"))
print("MAP Custom_Cosine =", calculate_map("Custom_Cosine"),"\n")

print("NDCG Audeering_Euclidean =", calculate_ndcg("Audeering_Euclidean"))
print("NDCG Audeering_Cosine =", calculate_ndcg("Audeering_Cosine"))
print("NDCG Custom_Euclidean =", calculate_ndcg("Custom_Euclidean"))
print("NDCG Custom_Cosine =", calculate_ndcg("Custom_Cosine"),"\n")

print("MRR Audeering_Euclidean =", calculate_mrr("Audeering_Euclidean"))
print("MRR Audeering_Cosine =", calculate_mrr("Audeering_Cosine"))
print("MRR Custom_Euclidean =", calculate_mrr("Custom_Euclidean"))
print("MRR Custom_Cosine =", calculate_mrr("Custom_Cosine"),"\n\n")

print("****** SPOTIGEM ******")
print("MAP Audeering_Euclidean =", calculate_map("Audeering_Euclidean", True))
print("MAP Audeering_Cosine =", calculate_map("Audeering_Cosine", True))
print("MAP Custom_Euclidean =", calculate_map("Custom_Euclidean", True))
print("MAP Custom_Cosine =", calculate_map("Custom_Cosine", True),"\n")

print("NDCG Audeering_Euclidean =", calculate_ndcg("Audeering_Euclidean", True))
print("NDCG Audeering_Cosine =", calculate_ndcg("Audeering_Cosine", True))
print("NDCG Custom_Euclidean =", calculate_ndcg("Custom_Euclidean", True))
print("NDCG Custom_Cosine =", calculate_ndcg("Custom_Cosine", True),"\n")

print("MRR Audeering_Euclidean =", calculate_mrr("Audeering_Euclidean", True))
print("MRR Audeering_Cosine =", calculate_mrr("Audeering_Cosine", True))
print("MRR Custom_Euclidean =", calculate_mrr("Custom_Euclidean", True))
print("MRR Custom_Cosine =", calculate_mrr("Custom_Cosine", True))

MAP Audeering_Euclidean = 0.7
MAP Audeering_Cosine = 0.7
MAP Custom_Euclidean = 0.6
MAP Custom_Cosine = 0.55 

NDCG Audeering_Euclidean = 0.7625293524551258
NDCG Audeering_Cosine = 0.8482673343981371
NDCG Custom_Euclidean = 0.8503582947546307
NDCG Custom_Cosine = 0.8228419422593065 

MRR Audeering_Euclidean = 0.41666666666666663
MRR Audeering_Cosine = 0.625
MRR Custom_Euclidean = 0.3625
MRR Custom_Cosine = 0.375 


****** SPOTIGEM ******
MAP Audeering_Euclidean = 0.6000000000000001
MAP Audeering_Cosine = 0.4
MAP Custom_Euclidean = 0.7
MAP Custom_Cosine = 0.7 

NDCG Audeering_Euclidean = 0.825828689249295
NDCG Audeering_Cosine = 0.6754148493331477
NDCG Custom_Euclidean = 0.89079257312557
NDCG Custom_Cosine = 0.8707029043729639 

MRR Audeering_Euclidean = 0.0
MRR Audeering_Cosine = 0.1
MRR Custom_Euclidean = 0.5
MRR Custom_Cosine = 0.0
